## Required imports

In [1]:
import numpy as np
import math
import torch
import random

---

## Defining graphing function for visualization and Value class

In [2]:
class Value:
    def __init__(self, data, _children = (), _op = (), label= ''):
        self.data = data
        self._prev = set(_children)
        self._backward = lambda: None
        self.grad = 0.0
        self._op = _op
        self.label = label
    
    def __repr__(self):
        return f"Value(data = {self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out
    
    def __sub__(self, other):
        return self + (-other)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f"**{other}")

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out._backward = _backward

        return out
    
    def __neg__(self):
        return self * -1

    def __truediv__(self, other): 
        return self * other**-1
    
    def __rmul__(self, other):
        return self * other
    
    def __radd__(self, other):
        return self + other
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward

        return out
    
    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self, ), "tanh")

        def _backward():
            self.grad += (1 - (t**2)) * out.grad

        out._backward = _backward

        return out
    

    def backward(self):

        self is self if isinstance(self, (Value)) else Value(self)

        self.grad = 1.0

        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        for node in reversed(topo):
            node._backward()

In [3]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

---

## Creating the classes to define the Neural Net

In [4]:
class Neuron:
    def __init__(self, inp):
        self.w = [Value(random.uniform(-1,1)) for _ in range(inp)] # for _ is used to repeat a block of code when not needing the loop
        self.b = Value(random.uniform(-1,1))                       # variable itself

    def __call__(self, x):
        activation = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = activation.tanh()

        return out
    
    def parameters(self):
        return self.w + [self.b] # returns list

class Layer:
    def __init__(self, inp, outp):
        self.neurons = [Neuron(inp) for _ in range(outp)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        params = []
        for neuron in self.neurons:
            ps = neuron.parameters()
            params.extend(ps) 
        return params
    
class MLP:
    def __init__(self, inp, outps):
        size = [inp] + outps # inp gets the number of input neurons and outps defines the rest of the structure
        self.layers = [Layer(size[i], size[i + 1]) for i in range(len(outps))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        params = []

        for layer in self.layers:
            params.extend(layer.parameters())

        return params

Class Neuron takes an input during initialization which defines the number of inputs going inside the neuron and solves them to return with a value calculated by the activation function. The activation function then undergoes tanh operation to give a value between -1 and 1.

Class Layer intakes an input and output value during initialization where the input value defines the number of values intaken by each neuron in the layer and output defines how many outputs are there from the layer, that is, the number of neurons in the layer itself.

Class MLP intakes an input VALUE and outputs LIST during initialization where the input tells the number of inputs in the MLP and the outputs defines the structure of the entire layer. Eg: outps = [3,2,1] => first hidden layer with 3 neurons, second hidden layer with 2 neurons and the final layer with 1 neuron that gives the output. 

---

## Working of the neural net on synthetic dataset

### Initializing the MLP

In [ ]:
neuralNet = MLP(3, [4,4,1]) # defining the MLP structure

xs = [
  [2.0, 3.0, -1.0], 
  [3.0, -1.0, 0.5], 
  [0.5, 1.0, 1.0], 
  [1.0, 1.0, -1.0],
] # 4x3 tensor

ys = [1.0, -1.0, -1.0, 1.0]

In [6]:
for iter in range(1000):
    yPrediction = [neuralNet(x) for x in xs]
    loss = sum([(youtp - yreal)**2 for yreal, youtp in zip(ys, yPrediction)])

    for p in neuralNet.parameters():
        p.grad = 0.0

    loss.backward()

    for p in neuralNet.parameters():
        p.data += -0.03 * p.grad

    # print(f"{iter}th iter => Loss: {loss.data}, Predictions: {yPrediction}")
print(yPrediction)

[Value(data = 0.98983190261718), Value(data = -0.9871127110502131), Value(data = -0.9852583771655935), Value(data = 0.9848961765428903)]


In [8]:
# draw_dot(loss)